# 💻 LOCAL: MIND Hallucination Detection (Intel Arc Optimized)

**Kernel**: Select `Python 3.11 (MIND - Intel Arc)` from the kernel picker (top-right)

### 🚀 What's Inside
- **Intel Arc GPU** via PyTorch 2.11.0+xpu
- **Secure keys** loaded from local `.env`
- **Multi-Layer Latent Dynamics** (40,960-dim features)
- **Automated GPT-4 evaluation** via OpenAI Batch API

In [14]:
# ═══════════════════════════════════════════════════════
# SHARED CONFIG — run this before any other cell
# ═══════════════════════════════════════════════════════
import os, subprocess

PROJECT_ROOT = r"c:\MSCS\Spring2026(ONE LAST DANCE)\CS6120\MIND"
PYTHON       = os.path.join(PROJECT_ROOT, ".venv_mind", "Scripts", "python.exe")
SRC          = os.path.join(PROJECT_ROOT, "src")
os.chdir(PROJECT_ROOT)

def run(script, *args):
    """Run a src/ script, streaming output line-by-line into the Jupyter cell."""
    cmd = [PYTHON, os.path.join(SRC, script)] + list(args)
    print(f"\n▶ {script} {' '.join(args)}")
    print('─' * 60)
    proc = subprocess.Popen(
        cmd, cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace'
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('─' * 60)
    if proc.returncode != 0:
        print(f"❌ {script} failed (exit code {proc.returncode})")
    else:
        print(f"✅ {script} completed")
    return proc.returncode

print("✅ Config loaded")
print(f"   PROJECT_ROOT = {PROJECT_ROOT}")
print(f"   PYTHON       = {PYTHON}")

✅ Config loaded
   PROJECT_ROOT = c:\MSCS\Spring2026(ONE LAST DANCE)\CS6120\MIND
   PYTHON       = c:\MSCS\Spring2026(ONE LAST DANCE)\CS6120\MIND\.venv_mind\Scripts\python.exe


## 🔍 Cell 1: Verify Everything (Re-run Anytime!)

In [15]:
import torch
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(PROJECT_ROOT, ".env"))

print("=" * 55)
print("   🧠 MIND Framework - Local Verification")
print("=" * 55)

try:
    xpu_ok = torch.xpu.is_available()
    name = torch.xpu.get_device_name(0) if xpu_ok else "N/A"
    print(f"\n🖥️  GPU: {'✅ ' + name if xpu_ok else '⚠️  No XPU - CPU only'}")
except Exception as e:
    print(f"\n🖥️  GPU: ⚠️  {e}")

hf  = os.getenv('HF_TOKEN')
oai = os.getenv('OPENAI_API_KEY')
print(f"\n🔑 HF_TOKEN:       {'✅ Loaded' if hf  else '❌ MISSING'}")
print(f"🔑 OPENAI_API_KEY: {'✅ Loaded' if oai else '❌ MISSING'}")

wiki = os.path.join(PROJECT_ROOT, "data", "auto-labeled", "wiki")
print("\n📁 Wikipedia Source Data:")
all_ok = True
for f in ["wiki_train.json", "wiki_valid.json", "wiki_test.json"]:
    ok = os.path.exists(os.path.join(wiki, f))
    print(f"   {f}: {'✅' if ok else '❌ MISSING'}")
    if not ok: all_ok = False

venv_ok = os.path.exists(PYTHON)
print(f"\n🐍 Venv Python:    {'✅' if venv_ok else '❌ NOT FOUND'}")
print(f"🔧 PyTorch:        {torch.__version__}")

print("\n" + "=" * 55)
print("✅ ALL CHECKS PASSED — Ready to Train!" if (all_ok and hf and oai and venv_ok)
      else "⚠️  Fix items marked ❌ above")
print("=" * 55)

   🧠 MIND Framework - Local Verification

🖥️  GPU: ✅ Intel(R) Arc(TM) Graphics

🔑 HF_TOKEN:       ✅ Loaded
🔑 OPENAI_API_KEY: ✅ Loaded

📁 Wikipedia Source Data:
   wiki_train.json: ✅
   wiki_valid.json: ✅
   wiki_test.json: ✅

🐍 Venv Python:    ✅
🔧 PyTorch:        2.11.0+xpu

✅ ALL CHECKS PASSED — Ready to Train!


## 🧪 Cell 2: Train the Hallucination Classifier
First run downloads Llama 3.1 8B (~16 GB). Expect 15-30 mins.

In [17]:
run("generate_data.py", "--model_family", "llama3base", "--model_type", "8", "--gpu", "0")
run("generate_hd.py",   "--model_family", "llama3base", "--model_type", "8", "--strategy", "multi_layer")
run("train.py", "--model_name", "llama3base8b", "--strategy", "multi_layer", "--device", "cuda:0")


▶ generate_data.py --model_family llama3base --model_type 8 --gpu 0
────────────────────────────────────────────────────────────
🚀 Intel Arc GPU detected — loading to XPU

Loading weights:  14%|█▎        | 40/291 [00:06<00:22, 11.10it/s]────────────────────────────────────────────────────────────
❌ generate_data.py failed (exit code 3221225477)

▶ generate_hd.py --model_family llama3base --model_type 8 --strategy multi_layer
────────────────────────────────────────────────────────────
Model: llama3base8b | Strategy: multi_layer


KeyboardInterrupt: 

## 📊 Cell 3: HELM Evaluation

In [ ]:
run("generate_helm_data.py", "--model_family", "llama3base", "--model_type", "8", "--gpu", "0")
run("label_helm_data.py", "--model_name", "llama3base8b", "--mode", "submit")

### ⏳ After OpenAI batch completes:

In [ ]:
run("label_helm_data.py",      "--model_name", "llama3base8b", "--mode", "fetch")
run("generate_hd_for_helm.py", "--model_family", "llama3base", "--model_type", "8", "--strategy", "multi_layer", "--gpu", "0")
run("detection_score.py",      "--strategy", "multi_layer", "--gpu", "0")

## 📈 Cell 5: Results Summary Table

In [ ]:
import pandas as pd

helm_dir = os.path.join(PROJECT_ROOT, "data", "helm")
print("=" * 65)
print("  📊 MIND Framework — Final Metrics (Multi-Layer Strategy)")
print("=" * 65)

metrics = {
    "Sent AUC (Halluc.)": "result_sent_halu_multi_layer.csv",
    "Sent Corr":          "result_sent_corr_multi_layer.csv",
    "Psg AUC (Halluc.)":  "result_psg_halu_multi_layer.csv",
    "Psg Corr":           "result_psg_corr_multi_layer.csv",
}

rows = []
for metric_name, filename in metrics.items():
    path = os.path.join(helm_dir, filename)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0)
        for model in df.index:
            rows.append({"Model": model, "Metric": metric_name,
                         "Score": round(float(df.loc[model, "Our_score"]), 2)})
    else:
        print(f"⚠️  {filename} not found — run detection_score.py first")

if rows:
    result_df = pd.DataFrame(rows).pivot(index="Model", columns="Metric", values="Score")
    print()
    print(result_df.to_string())
    print()
    print("✅ Strategy : multi_layer | Features: 40,960-dim (5-layer concat + deltas)")
    print("📌 Higher AUC = better hallucination detection")
    print("📌 Higher Corr = better passage-level calibration")
else:
    print("No results yet — run Cells 2–4 first.")